# First GRPO Experiment

**Goal:** get GRPO working end-to-end on a small model and a single-turn math task, using an outcome-only (verifiable) reward. This is Stage 0 of the plan — the point is not a good result, it's watching the reward curve move and trusting your own pipeline.


**Model:** Qwen2.5-0.5B-Instruct (small enough to train comfortably on a free T4)

**Dataset:** GSM8K (grade-school math word problems with a clean final numeric answer — easy to verify programmatically)

**Method:** LoRA (so we only train a small adapter, not the full model) + TRL's GRPOTrainer

## 1. Install dependencies

In [1]:
!pip install -q trl peft transformers datasets accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.2 MB/s eta 0:00:00


## 2. Load the model and tokenizer

We use LoRA so we're only training a small set of adapter weights on top of the frozen base model — this is what makes training feasible on a free-tier GPU.

In [3]:
!pip uninstall -y torchao -q

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see a tiny fraction of total parameters listed as trainable -
# that's the whole point of LoRA: everything else stays frozen.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


## 3. Load and prepare GSM8K

Each example is a grade-school math word problem with a final numeric answer marked after `####` in the original dataset. We reformat this as a chat-style prompt.

In [5]:
from datasets import load_dataset
import re

dataset = load_dataset("openai/gsm8k", "main", split="train")

SYSTEM_PROMPT = (
    "You are a careful math tutor. Solve the problem step by step, "
    "then give the final numeric answer on its own line in the form: "
    "Answer: <number>"
)

def extract_ground_truth(example):
    # GSM8K answers look like: "... some reasoning #### 42"
    match = re.search(r"####\s*(-?[0-9,.]+)", example["answer"])
    gt = match.group(1).replace(",", "") if match else None
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "ground_truth": gt,
    }

dataset = dataset.map(extract_ground_truth)
dataset = dataset.filter(lambda x: x["ground_truth"] is not None)

# Use a small subset for a first run - this is about watching the pipeline
# work, not squeezing out maximum performance.
dataset = dataset.select(range(500))
print(dataset[0])


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7473 [00:00<?, ? examples/s]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'prompt': [{'role': 'system', 'content': 'You are a careful math tutor. Solve the problem step by step, then give the final numeric answer on its own line in the form: Answer: <number>'}, {'role': 'user', 'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}], 'ground_truth': '72'}


## 4. Define the reward function

This is a purely verifiable, rule-based reward: does the model's final numeric answer match the ground truth? No learned reward model, no ambiguity, this is RLVR in its simplest form.

TRL's GRPOTrainer calls your reward function with the full list of completions (one batch = your group of sampled responses per prompt) and expects a list of scalar rewards back.

In [6]:
def extract_predicted_answer(text):
    match = re.search(r"Answer:\s*(-?[0-9,.]+)", text)
    if match:
        return match.group(1).replace(",", "").rstrip(".")
    return None

def correctness_reward(completions, ground_truth, **kwargs):
    """
    completions: list of generated responses (strings) for this batch
    ground_truth: list of correct answers, aligned with completions
    Returns: list of floats, one reward per completion
    """
    rewards = []
    for completion, gt in zip(completions, ground_truth):
        text = completion if isinstance(completion, str) else completion[-1]["content"]
        predicted = extract_predicted_answer(text)
        if predicted is None:
            rewards.append(0.0)  # no parseable answer at all
        else:
            try:
                correct = abs(float(predicted) - float(gt)) < 1e-4
            except ValueError:
                correct = False
            rewards.append(1.0 if correct else 0.0)
    return rewards

def format_reward(completions, **kwargs):
    """
    Small bonus for actually following the requested output format.
    This is a taste of reward *shaping* - and also a taste of how naive
    hand-designed rewards can be gamed if you're not careful (something
    worth watching for once you inspect real outputs).
    """
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[-1]["content"]
        rewards.append(0.1 if re.search(r"Answer:\s*-?[0-9,.]+", text) else 0.0)
    return rewards


## 5. Configure and run GRPO

Key hyperparameters to recognize from what you already know:
- `num_generations`: this is your group size — how many responses get sampled per prompt before computing the group-relative advantage. Start small (4-8).
- `beta` (KL coefficient): keeps the policy from drifting too far from the reference model, the reward-hacking mitigation

In [8]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir="./grpo-gsm8k-qwen0.5b",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    num_train_epochs=1,
    beta=0.04,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward],
    args=config,
    train_dataset=dataset,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.095970
10,0.053079
15,0.068934
20,0.080790
25,0.116307
30,0.064934
35,0.072045
40,0.077050
45,-0.013858
50,0.028463


Step,Training Loss
5,0.095970
10,0.053079
15,0.068934
20,0.080790
25,0.116307
30,0.064934
35,0.072045
40,0.077050
45,-0.013858
50,0.028463


TrainOutput(global_step=125, training_loss=0.053466733694076535, metrics={'train_runtime': 3728.1442, 'train_samples_per_second': 0.134, 'train_steps_per_second': 0.034, 'total_flos': 0.0, 'train_loss': 0.053466733694076535, 'epoch': 1.0})

## 6. Inspect results


In [12]:
test_dataset = load_dataset("openai/gsm8k", "main", split="test").select(range(10))

for example in test_dataset:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    match = re.search(r"####\s*(-?[0-9,.]+)", example["answer"])
    gt = match.group(1).replace(",", "") if match else "?"

    print("QUESTION:", example["question"][:150])
    print("MODEL RESPONSE:", response)
    print("GROUND TRUTH:", gt)
    print("-" * 80)

QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the rem
MODEL RESPONSE: Step-by-align-block"> code syntax

    let `  �、“3、《关于    �/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T/T
GROUND TRUTH: 18
--------------------------------------------------------------------------------
QUESTION: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
MODEL RESPONSE: Answered code/API/API/API/API/API/API/API/API/API/API/API/API/API

In [ ]:
# Load a fresh, untrained copy of the base model and test generation on it directly,
# bypassing your LoRA adapter entirely, to see if the garbage exists BEFORE any RL training.
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Janet's ducks lay 16 eggs per day. She eats three for breakfast and bakes muffins with four. She sells the rest at $2 each. How much does she make daily?"},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(base_model.device)
output = base_model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

First, we calculate the total number of eggs Janet's ducks lay in one day:
\[
16 \text{ eggs/day}
\]

Next, we subtract the number of eggs she eats as breakfast and the number of muffins she bakes from this amount:
\[
16 \text{ eggs} - 3 \text{ eggs (breakfast)} - 4 \text{ muffins} = 7 \text{ eggs/day}
\]

Now, we determine how many eggs she sells at $2 each:
\[
7 \text{ eggs/day} \times \$2/\text{egg} = \$14 \text{ per day}
\]

Thus, Janet makes \(\boxed{14}\) dollars daily.


## Run 1 Results Summary (recorded after running)

Outcome: Training completed (125 steps,`train_loss=0.053`), but the trained model produced severely degenerate output on held-out test questions — repetitive token loops (e.g. endless `/API/API/API...`) instead of coherent reasoning.

Control check: A fresh, untrained copy of the base model produced fully coherent step-by-step reasoning on the same question. This confirms the degeneration was induced by this GRPO run, not a pre-existing model/code issue.

Diagnosed likely cause (hypothesis, not yet confirmed):
- `num_generations=4` is a small group size for a genuinely hard task (GSM8K on a 0.5B model) — plausibly many groups had identical reward across all 4 completions (likely all wrong), giving zero group standard deviation.
- GRPO's advantage is `(reward - group_mean) / group_std`. Zero std breaks this into a degenerate or unstable signal instead of a meaningful update.
- Small, noisy `train_loss` values throughout (no clear trend) are consistent with mostly-uninformative updates punctuated by occasional poorly-scaled ones.
- The shallow `format_reward` (bonus for the literal string "Answer: <number>") may have dominated the little learning signal that existed, since it's trivially easy to satisfy compared to the sparse correctness signal — a plausible, concrete instance of reward hacking.

ot yet tested / next steps:
- Did not log or inspect `reward`, `reward_std`, `kl`, or `entropy` during training (only `train_loss`) — the most important fix for next run, since these would directly confirm/refute the zero-variance hypothesis.
- Did not check the base model's raw pre-RL accuracy on the training subset (unknown how sparse the correctness reward actually was).
- Next run: increase `num_generations` (e.g. 4 -> 8), add full metric logging, and consider dropping/down-weighting `format_reward` as an ablation.

Research connection: A concrete, hands-on demonstration that naive reward design can cause severe unintended behavior even in the simplest single-turn RLVR setup, before any multi-turn/agentic complexity is added — directly relevant to the credit-assignment/reward-hacking research direction.
